# ClashCompare V3.3 — Entraînement IA
Ce notebook prépare un détecteur de bâtiments puis exporte le modèle en ONNX pour ClashCompare.

**Important :** il faut d'abord ajouter et annoter des images dans `training/dataset/detector`.

In [ ]:
!pip -q install ultralytics


## 1. Monter Google Drive (option pratique depuis iPhone)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Modifie ce chemin pour pointer vers le dossier ClashCompare-V3.3 dans ton Drive
PROJECT = '/content/drive/MyDrive/ClashCompare-V3.3-AI-Training-Ready'


## 2. Vérifier le dataset


In [ ]:
from pathlib import Path
root = Path(PROJECT)
train_images = list((root/'training/dataset/detector/images/train').glob('*.*'))
val_images = list((root/'training/dataset/detector/images/val').glob('*.*'))
print('Images train:', len(train_images))
print('Images val:', len(val_images))
assert len(train_images) > 0, 'Ajoute des images annotées avant de lancer l’entraînement.'


## 3. Entraîner le détecteur


In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
results = model.train(
    data=str(root/'training/dataset.yaml'),
    epochs=80,
    imgsz=1024,
    batch=8,
    project=str(root/'training/runs'),
    name='building-detector',
    patience=15
)


## 4. Exporter pour le navigateur


In [ ]:
best = root/'training/runs/building-detector/weights/best.pt'
export_model = YOLO(str(best))
export_model.export(format='onnx', imgsz=1024, simplify=True, dynamic=False)
print('Copie ensuite best.onnx dans le dossier models/ de ClashCompare.')
